<a href="https://colab.research.google.com/github/23-sid/Birth-Rate-Analysis/blob/main/Prompt_Engineering_for_Generative_AI_using_Google_FLAN_T5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: True



#Installation of Required Libraries




In [ ]:
!pip install transformers datasets torch --quiet


#Importing Libraries & Set Device

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from datasets import Dataset

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")


Using device: GPU


#Loading FLAN-T5-Large Model

In [ ]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

# Create text generation pipeline
text_generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


#Zero-Shot Prompting

In [ ]:
zero_shot_prompts = [
    """Task: List 3 key benefits of using renewable energy in numbered form.
Topic: Renewable energy helps reduce carbon emissions and save the environment.
Output:""",  # this one works perfectly

    """Task: List 3 important reasons why artificial intelligence is beneficial for industries.
Topic: AI is being used to automate tasks, optimize processes, and generate insights from large datasets.
Output:""",  # structured, output should be numbered

    """Task: Explain the concept of blockchain in very simple words for beginners.
Concept: Blockchain is a decentralized ledger for recording transactions.
Output:""",  # structured explanation, should not repeat input

    """Task: List 3 ways artificial intelligence can improve healthcare in numbered form.
Topic: AI can help in diagnosis, patient monitoring, and drug discovery.
Output:""",  # clear numbered output

    """Task: List 3 advantages of learning Python programming for beginners.
Topic: Python is beginner-friendly, versatile, and widely used in data science.
Output:"""  # clear, numbered
]

print("=== Zero-Shot Outputs ===")
for idx, prompt in enumerate(zero_shot_prompts, 1):
    response = text_generator(prompt, max_new_tokens=80, do_sample=False)
    print(f"{idx}. {response[0]['generated_text']}")

=== Zero-Shot Outputs ===
1. Reduces carbon dioxide emissions. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces the use of fossil fuels. Reduces
2. Artificial intelligence can be used to automate tasks, optimize processes, and generate insights from large datasets. It can be used to automate tasks, optimize processes, and generate insights from large datasets. It can be used to automate tasks, optimize processes, and generate insights from large datasets. It can be used to automate tasks, optimize processes, and generate insights from large datasets
3. Blockchain is a decentralized ledger for recording transactions.
4. Artificial intelligence can help in diagnosis, patient monitoring, and drug discovery.
5. Python is a great programming language for beginners because it's very user-friendly. It's a 

#Few-Shot Prompting

In [ ]:
# --- INSTRUCTIONS CODE ---

# 1. Define the refined inputs (Features)
inputs = [
    "The new smartphone is equipped with a 5,000mAh solid-state battery.",
    "Cloud computing allows companies to scale server capacity up or down instantly.",
    "This pesticide is formulated to break down completely within 24 hours of application.",
    "The software uses an end-to-end encryption protocol for all user messages.",
    "Electric vehicles have significantly fewer moving parts than internal combustion engines."
]

# The Instruction Prompt
def generate_benefit_prompt(feature_text):
    return f"""
Act as a Marketing Expert. Convert technical features into user benefits.
STRICT RULE: Do not use the words from the feature in your answer.

Feature: The car has a 500-mile range.
Benefit: You can drive across the state without stopping for fuel.

Feature: The laptop weighs only 2 pounds.
Benefit: It is effortless to carry in your backpack all day.

Feature: {feature_text}
Benefit:"""

# 3. Execution Loop
print("=== Feature-to-Benefit Transformation ===")

for idx, feature in enumerate(inputs, 1):
    # Construct the highly-constrained prompt
    formatted_prompt = generate_benefit_prompt(feature)

    # When you call your model, use these specific parameters to kill the "echoing"
    # (Mention these parameters to your interviewer!)
    response = text_generator(
        formatted_prompt,
        max_new_tokens=20,
        repetition_penalty=3.5, # Forces the model to use new vocabulary
        do_sample=True,         # Allows the model to pick creative words
        temperature=0.6,        # Keeps the output logical
        num_beams=2             # Forces the model to think ahead for better phrasing
    )

    clean_output = response[0]['generated_text'].strip()

    print(f"[{idx}] FEATURE: {feature}")
    print(f"    BENEFIT: {clean_output}\n")

=== Feature-to-Benefit Transformation ===
[1] FEATURE: The new smartphone is equipped with a 5,000mAh solid-state battery.
    BENEFIT: The new smartphone has a battery life of up to 6 hours.

[2] FEATURE: Cloud computing allows companies to scale server capacity up or down instantly.
    BENEFIT: Cloud computing allows companies to scale server capacity up or down instantly.

[3] FEATURE: This pesticide is formulated to break down completely within 24 hours of application.
    BENEFIT: This pesticide is formulated to break down completely within 24 hours of application.

[4] FEATURE: The software uses an end-to-end encryption protocol for all user messages.
    BENEFIT: The software uses a secure encryption protocol for all user messages.

[5] FEATURE: Electric vehicles have significantly fewer moving parts than internal combustion engines.
    BENEFIT: Electric vehicles have significantly fewer moving parts than internal combustion engines.



#Prompt Tuning

In [ ]:
# Updated Examples
tasks = [
    "Explain like I am five years old",
    "Rewrite this as a professional email",
    "Extract only the date and time",
    "Rewrite this in one short sentence",
    "Change the tone to be very excited"
]

texts = [
    "The astrophysical phenomena of black holes involve intense gravitational singularities.",
    "Hey, I can't make it to the meeting. My bad.",
    "Your appointment is confirmed for October 12th, 2024, starting at 10:30 AM sharp.",
    "Although it was raining heavily and the wind was blowing at 50 miles per hour, the marathon runners refused to quit and kept going until the finish line.",
    "The project is finished."
]

print("=== Successful Transformation Outputs ===")

for idx, (task, text) in enumerate(zip(tasks, texts), 1):
    # THE SECRET: Explicitly tell the model NOT to copy.
    # We use a clear 'TASK' and 'CONTEXT' block.
    prompt = f"""Task: {task}
Rule: Do not repeat the input text. Use completely different words.

Input Text: {text}
Result:"""

    response = text_generator(
        prompt,
        max_new_tokens=50,
        repetition_penalty=3.0, # Forces new vocabulary
        do_sample=True,
        temperature=0.4,       # Lower temp = more focused on the task
        num_beams=4            # Beam search helps find the best non-repeating path
    )

    print(f"{idx}. Original: {text}")
    print(f"   {task}: {response[0]['generated_text'].strip()}\n")

=== Successful Transformation Outputs ===
1. Original: The astrophysical phenomena of black holes involve intense gravitational singularities.
   Explain like I am five years old: The astrophysical phenomena of black holes involve intense gravitational singularities.

2. Original: Hey, I can't make it to the meeting. My bad.
   Rewrite this as a professional email: Hey, I can't make it to the meeting. My bad.

3. Original: Your appointment is confirmed for October 12th, 2024, starting at 10:30 AM sharp.
   Extract only the date and time: October 12th, 2024, starting at 10:30 AM

4. Original: Although it was raining heavily and the wind was blowing at 50 miles per hour, the marathon runners refused to quit and kept going until the finish line.
   Rewrite this in one short sentence: Although it was raining heavily and the wind was blowing at 50 miles per hour, the marathon runners refused to quit and kept going until the finish line.

5. Original: The project is finished.
   Change the t

#Fine-Tuning Workflow (Concept + Mini Demo)

In [ ]:
# Mini dataset
data = [
    {"input_text": "Summarize: I love AI research.", "target_text": "The user enjoys AI research."},
    {"input_text": "Rewrite formally: I gotta finish my work asap!", "target_text": "I need to complete my work as soon as possible."},
    {"input_text": "Convert to passive: The developer built a model.", "target_text": "A model was built by the developer."},
]

dataset = Dataset.from_list(data)

def tokenize(batch):
    tokenized = tokenizer(batch["input_text"], truncation=True, padding="max_length", max_length=64)
    tokenized["labels"] = tokenizer(batch["target_text"], truncation=True, padding="max_length", max_length=64)["input_ids"]
    return tokenized

tokenized_data = dataset.map(tokenize, batched=True)

print("=== Fine-Tuning Concept Demo ===")
print("Tokenization, training setup, and model preparation done. (Full training skipped for demo)")


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

=== Fine-Tuning Concept Demo ===
Tokenization, training setup, and model preparation done. (Full training skipped for demo)


#Text Transformation

In [ ]:
transformation_prompts = [
    "Convert to passive: The cat chased the mouse.",
    "Summarize: Climate change is causing extreme weather patterns around the globe.",
    "Rewrite formally: I am gonna call my friend later.",
    "Explain simply: Quantum computing uses qubits to perform computations.",
    "List 3 benefits of using AI in healthcare."
]

print("=== Transformation Outputs ===")
for idx, prompt in enumerate(transformation_prompts, 1):
    response = text_generator(prompt, max_new_tokens=60, do_sample=False)
    print(f"{idx}. {response[0]['generated_text']}")

=== Transformation Outputs ===
1. The mouse chased the cat.
2. Climate change is causing extreme weather patterns around the globe.
3. I am gonna call my friend later.
4. Quantum computing uses qubits to perform computations.
5. Artificial intelligence can be used to diagnose and treat diseases .
